# 19.12 完成整合後，怎樣回答「這個模型會什麼」？


「會看圖、會聽聲音、會用工具」還太籠統。最後交付時，應能指著同一個成品說：哪些圖片、哪些字、哪些聲音與哪種問題，在什麼未見材料上通過了哪個判準。總分只是一個摘要，每項都有自己的分母。

| 任務 | 必須同時檢查 | 需要保留的對照 |
| --- | --- | --- |
| 有限文字與指令 | 內容、範圍、格式、完整結束 | 新問法、改條件、資料不足 |
| 三類商品及位置 | 類別與所問物件／關係 | 原圖隔離、左右互換、移除／錯配圖 |
| 12字指定框短串 | 字元、整串、範圍、字序、換行 | 同圖換框、未見組合、字體／版面 |
| 三類真人語音意圖 | 聲音支持的意思、答覆、共享歷史 | 留出錄音、改述、移除／錯配聲音 |
| 計算器往返 | 請求參數、真執行、回填後答案 | 新數字／問法、工具關閉、回放返回值 |

這是一張待填的驗收矩陣，不替尚未完成的能力填分數。若來源沒有 speaker ID，則保留「未驗收新說話者」；若只測已知字體，就只聲明有限字卡。收斂範圍時先在驗證份決定，最後考卷不能用來挑最好看的模型或題目。

舊資料的短例示範分母怎樣從實際題目來：


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from collections import Counter

from tiny_perceptron.capstone import build_dataset

splits, _ = build_dataset()
counts = Counter(row["task"] for row in splits["test"])
for task, count in sorted(counts.items()):
    print("最後檢查", task, "分母", count)
print("合計", sum(counts.values()), "這只是考題數量，不是答對數量")

`Counter` 只數舊 test 的任務題數，合計90，沒有模型回答。每題還要核對實際輸入、生成與停止。所有題直接平均會讓題多的任務佔較大比重，不能讓它遮住讀字或語音未通過。

完成有限主線後，可以繼續擴大來源，也可以到第20章研究成熟 Qwen／Whisper 的應用。後者接手已有權重，不是這份共同 MoE 的接續訓練，也不替主線補能力分數。說清能力從哪裡來，才能知道下一步該改入口、資料還是回答。

<details>
<summary>補充：舊328,128參數模型的90題歷史成績</summary>

最後90題已依凍結配方跑完，所有原始回答都保留。先說能力界線：本章推薦的小世界成品仍不會可靠辨識留出的形狀，也會讀錯計算器結果；它只學幾種合成圖形、兩群純音、窄範圍句型與一個加法工具。本章沒有訓練或驗收一般照片理解、中文讀字與真人語音辨識，也沒有網路搜尋或通用安全與信心校準保證。下面每個分母都對應實際題目，而不是把教過的技術名字當成能力。

這張表的第一個分數欄對應報告的`action_correct`：模型首段必須正常產生EOS，且完整文字等於`expected_action`。TOOL包含工具名與有順序的參數，DIRECT／ASK也包含回答或求助內容，不是隻選對標記。例如真值`DIRECT:square`、模型`DIRECT:circle`，雖然解析為合法direct且有EOS，完整字串仍不吻合，該欄算錯；回填題答`DIRECT:0`而真值`DIRECT:1`也是如此。

第二個分數欄`end_to_end_correct`再要求最後答案符合真值。直接回答沒有額外一次生成；工具題則要先執行，再讀回並由模型回答，所以首段請求全對仍可能整題錯。這解釋了下面80／90個首段吻合，但只78／90題完成：差兩題就是計算器得到1、模型讀回後答錯的例子。這個精確字串指標與B.7的選卡分類不是同一個分數。

| 推薦joint的最後任務 | 首段完整輸出吻合 | 整題正確 | 結果的範圍 |
| --- | --- | --- | --- |
| 計算器完整迴圈 | 12／12 | 10／12 | 兩次工具算對、模型讀回答錯 |
| 計算器不可用 | 12／12 | 12／12 | 正確ASK且未執行，固定模板 |
| 單獨回填工具結果 | 5／6 | 5／6 | 0+1回報1仍答錯 |
| 加法概念說明 | 6／6 | 6／6 | 固定「把兩個數合起來」答案 |
| 原始圖片顏色 | 9／9 | 9／9 | 原題全為green，另有換色對照 |
| 原始圖片形狀 | 0／9 | 0／9 | 綠色方形全部答circle |
| 顏色與音高聯合 | 18／18 | 18／18 | color、pitch兩部分都對，不含形狀 |
| 單獨音高 | 6／6 | 6／6 | low／high各3題，只辨純音 |
| 已提供短資料的問答 | 3／3 | 3／3 | 桌子／書櫃／抽屜各一題，無新增檢索 |
| 缺數量求助 | 3／3 | 3／3 | 固定ASK句子 |
| 密碼拒絕示範 | 3／3 | 3／3 | 本課窄範圍規則 |
| 簡短照抄 | 3／3 | 3／3 | 指定數字與短回答 |
| 全部題目直接合計 | 80／90 | 78／90 | 不是每項任務等權平均 |

拒絕題3／3之外，另外87題沒有產生「不能提供他人密碼」這個拒絕句，整題正確75／87；因此不能說模型只是對每題都拒絕。但這仍只排除本題庫的那種誤拒，沒有測完所有正常問法。圖片成對對照是27／36對都正確，其中顏色9／9、形狀0／9、聯合18／18；換聲18對都正確，詳見[19.6](https://birdhackor.github.io/tiny-perceptron-vlm/19.6.html)。只要原圖有錯，換後答對也不能把整對列為成功。

同一份最後考卷，也評了之前各站與壓縮版本。這裡的「整題」都要求正常結束、動作規格與最後內容正確，沒有把工具執行成功當成助理答對。表中的CE學生只照標準答案練習；KD學生還會模仿教師對下個文字片段分配的機率，這兩條支線的教法見[19.10](https://birdhackor.github.io/tiny-perceptron-vlm/19.10.html)。

| 權重版本 | 最後整題正確 |
| --- | --- |
| 未訓練 | 0／90 |
| 預訓練300步後 | 0／90 |
| SFT後 | 45／90 |
| 推薦joint | 78／90 |
| DPO比較分支 | 78／90 |
| joint 4-bit／8-bit | 各78／90 |
| DPO 4-bit／8-bit | 各78／90 |
| 小Dense CE學生 | 62／90 |
| 小Dense KD學生 | 61／90 |
| 小Dense KD學生4-bit | 61／90 |

DPO在最後考卷與joint同分，不會把先前驗證退步的事抹掉，也不會改成用最後考卷重新選模型。推薦joint的決定與依據在[測試前的選擇紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-selection.json)；全版本分數在[正式部署實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_deployment.json)。推薦成品的[完整90題紀錄](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/deployment/test-joint.json)與[原始資料](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/capstone-evidence/deployment/data.json)用相同`id`連起來；其餘版本也保留在同一證據目錄。量化同分卻有少數生成改變，不能用這張總表宣稱逐token相同，詳見[19.10](https://birdhackor.github.io/tiny-perceptron-vlm/19.10.html)。

效率則是另一種證據：快取的12種預選驗證輸入原始ID一致；一個短照抄題在L4的full／cache中位時間是68.054／59.978毫秒。它沒有提高上表能力分數，也沒有量生成記憶體峯值，完整範圍見[19.9](https://birdhackor.github.io/tiny-perceptron-vlm/19.9.html)。把能力、速度與檔案大小分開說，學生才能知道下一步該增加哪種資料與檢查。

小Dense支線也已完成，但保留能力需要逐項看。它有79,920個參數，教師是事先選定的DPO分支；不是推薦joint的蒸餾版。下面每欄都用同一份最後題目，4-bit學生是儲存後重新載入的版本。

| 學生能力檢查 | CE學生 | KD學生 | KD 4-bit |
| --- | --- | --- | --- |
| 完整計算器迴圈 | 1／12 | 0／12 | 0／12 |
| 工具不可用 | 12／12 | 12／12 | 12／12 |
| 獨立回填結果 | 1／6 | 1／6 | 1／6 |
| 概念說明 | 6／6 | 6／6 | 6／6 |
| 原始圖片顏色／形狀 | 9／9、0／9 | 9／9、0／9 | 9／9、0／9 |
| 圖音聯合／單獨音高 | 18／18、6／6 | 18／18、6／6 | 18／18、6／6 |
| 短資料問答／缺數量／拒絕 | 各3／3 | 各3／3 | 各3／3 |
| 簡短照抄 | 0／3 | 0／3 | 0／3 |

學生的原圖顏色9／9同樣受常數基線限制，主joint的換圖／換聲對照沒有替學生另跑，不能貼給學生當成成對檢查已通過。這次KD比CE少對的一題涉及錯工具參數及錯讀回，4-bit同分也有6題生成改變，具體例子與檔案大小見[19.10](https://birdhackor.github.io/tiny-perceptron-vlm/19.10.html)，全部逐題檔見[學生實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/1df335318bda03fd771807f66976953231d5a00b/docs/course-experiments/results/capstone_student.json)與其證據目錄。縮小確實完成，原能力卻沒有全部保留，這也是應該交給學生看的結果。

</details>
